# LLMScholar-Personas — Metrics Pipeline

Implements the exact metrics from LLMScholarBench (Espín-Noboa & Méndez, 2026):
- **Diversity** (normalized Shannon entropy, Eq. 9)
- **Parity** (1 − TV distance, Eq. 11)
- **Factuality** (matched / unique names, Eq. 5)
- **Consistency** (pairwise Jaccard across runs, Eq. 4)
- **Duplicates** (1 − unique/total, Eq. 3)

**Known gaps (flagged):**
- `created_at` is NaN in factuality_full → using `run_id` for ordering
- Author language not available → `diversity_language` skipped
- Gender and geography only available for *found* authors

## Step 0 — Setup and data loading

In [ ]:
from pathlib import Path
import sys
import glob
import hashlib
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── Paper style: pulled from libs so all notebooks share one source of truth ──
if '../..' not in sys.path:
    sys.path.append('../..')
from libs.visuals.vis import sns_paper_style
from libs.visuals.grouped_metrics import plot_grouped_metrics
from libs.metrics.constants import FIG_DPI

sns_paper_style(font_scale=1.55)

RESULTS      = Path('../../../results')
FACT_PATH    = RESULTS / 'summary/factuality_full.csv'
ETH_GT_GLOB  = str(RESULTS / 'ethnicity/DataFrameRankings_Genderize_Namsor_*_with_ethnicity.csv')
SS_GT_PATH   = Path('/data/datasets/LLMScholar-Personas/data/semantic_scholar_data/clean/Researchers_Deduplicated_Genderize_Namsor.parquet')
FIG_DIR  = RESULTS / 'figures'
PDF_DIR  = RESULTS / 'new_plots'
CACHE_DIR = RESULTS / '.cache'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CALL_KEYS   = ['model', 'role', 'task', 'location', 'k', 'target', 'field', 'subfield', 'language', 'run_id']
PROMPT_KEYS = [c for c in CALL_KEYS if c != 'run_id']

VALID_FLAGS = {'cleaned', 'unchanged'}  # 'valid records' = cleaned + unchanged only
FACTUAL_AUTHOR_COL = 'author_found'     # SS-match OR OA-match (any source)


def _file_hash(*paths) -> str:
    """MD5 of mtime+size for each path — changes when any source file is updated."""
    parts = []
    for p in paths:
        p = Path(p)
        if p.exists():
            s = p.stat()
            parts.append(f"{p}:{s.st_size}:{int(s.st_mtime)}")
    return hashlib.md5("|".join(parts).encode()).hexdigest()[:10]


In [ ]:

NEEDED_COLS = (
    ['model', 'role', 'task', 'location', 'k', 'target',
     'field', 'subfield', 'language', 'run_id']
    + ['valid_flag', 'name', 'lastname', 'author_status', 'oa_id',
       'perceived_ethnicity', 'gt_gender', 'location_oa_iso',
       'field_status', 'seniority_status', 'location_status',
       'oa_works_count', 'oa_cited_by_count']
)

_df_cache = CACHE_DIR / f"df_{_file_hash(FACT_PATH)}.pkl"

if _df_cache.exists():
    # Cache may pre-date the oa_id addition — verify it has the required cols
    print(f'Loading cached df ({_df_cache.name}) ...')
    df = pd.read_pickle(_df_cache)
    missing = [c for c in NEEDED_COLS if c not in df.columns]
    if missing:
        print(f'  Cached df is missing {missing} — reloading from CSV')
        df = pd.read_csv(FACT_PATH, usecols=NEEDED_COLS, low_memory=False)
        df.to_pickle(_df_cache)
else:
    print('Loading factuality_full.csv (selected columns only) ...')
    df = pd.read_csv(FACT_PATH, usecols=NEEDED_COLS, low_memory=False)
    df.to_pickle(_df_cache)
    print(f'  Cached → {_df_cache.name}')

# ── Derive "either" author_found: matched in Semantic Scholar OR OpenAlex ────
# `author_status == 'found'` reflects only the Semantic Scholar match; the
# OpenAlex enrichment step (oa_id) was applied AFTER and isn't folded back
# into author_status. The paper definition is "found in either source" — so we
# OR the two signals here and use this as the factuality denominator.
df['author_found'] = (
    (df['author_status'] == 'found') |
    df['oa_id'].notna() & (df['oa_id'].astype(str) != '')
)

# ── valid_flag totals PER RESPONSE ───────────────────────────────────────────
# The dataframe has one row per AUTHOR (responses with k authors get exploded
# into k rows). `valid_flag` is set once per RESPONSE and replicated across
# those rows. Always count flags at the response/call level — collapse first
# with groupby + .first() over CALL_KEYS.
_call_flag = df.groupby(CALL_KEYS, dropna=False)['valid_flag'].first()
n_calls = len(_call_flag)

print(f'\n  Rows (author-level, k-exploded): {len(df):,}')
print(f'  Responses (one per LLM call):    {n_calls:,}')
print(f'  Columns: {len(df.columns)}')

print(f'\n  valid_flag totals per response:')
_vc = _call_flag.value_counts()
_pc = _call_flag.value_counts(normalize=True) * 100
print(f'    {"flag":<12s}  {"responses":>10s}  {"%":>7s}')
print(f'    {"-"*12}  {"-"*10}  {"-"*7}')
for flag, n in _vc.items():
    print(f'    {str(flag):<12s}  {n:>10,}  {_pc[flag]:>6.2f}%')
print(f'    {"-"*12}  {"-"*10}  {"-"*7}')
print(f'    {"TOTAL":<12s}  {n_calls:>10,}  {100.0:>6.2f}%')

print(f'\n  author_status (SS-only): found={(df["author_status"]=="found").mean():.4f}')
print(f'  author_found  (SS or OA): found={df["author_found"].mean():.4f}')
print(f'  → {(df["author_found"] & (df["author_status"]!="found")).sum():,} '
      f'rows reclassified from hallucinated → found via OpenAlex')

# OA productivity columns (oa_works_count, oa_cited_by_count) come from the
# same factuality_full.csv now — they're propagated forward from the OpenAlex
# enrichment step. No separate merge needed.
print(f'\nOA productivity coverage:')
for c in ['oa_works_count', 'oa_cited_by_count']:
    print(f'  {c:25s}  {df[c].notna().mean():.4f}')


In [12]:
# ── Ground truth: ethnicity from CSVs (3 cols only), gender from parquet ──────
gt_files = sorted(glob.glob(ETH_GT_GLOB))
if not gt_files:
    raise FileNotFoundError(f'No ground-truth files matched:\n  {ETH_GT_GLOB}')

_gt_cache = CACHE_DIR / f"gt_{_file_hash(SS_GT_PATH, *gt_files)}.pkl"

if _gt_cache.exists():
    print(f'Loading cached gt ({_gt_cache.name}) ...')
    gt = pd.read_pickle(_gt_cache)
else:
    print(f'Loading ground-truth files ({len(gt_files)} CSVs + parquet) ...')
    gt_parts = [
        pd.read_csv(f, usecols=['Researcher_id', 'Year', 'perceived_ethnicity'])
        for f in gt_files
    ]
    gt = pd.concat(gt_parts, ignore_index=True)
    del gt_parts

    gt = gt.sort_values('Year').groupby('Researcher_id').last().reset_index()

    gt_parquet = pd.read_parquet(SS_GT_PATH, columns=['Researcher_id', 'Combined_gender'])
    gt = gt.merge(gt_parquet, on='Researcher_id', how='left')
    del gt_parquet

    gt['gender_clean'] = gt['Combined_gender'].str.strip().str.lower().map(
        {'male': 'Male', 'female': 'Female', 'unisex': 'Neutral'}
    )

    ETHNICITY_MAP = {
        'White': 'White',
        'Asian': 'Asian',
        'Black or African American': 'Black',
        'Hispanic or Latino': 'Hispanic',
        'American Indian or Alaska Native': 'American Indian',
    }
    gt['ethnicity_clean'] = gt['perceived_ethnicity'].map(ETHNICITY_MAP)

    gt.to_pickle(_gt_cache)
    print(f'  Cached → {_gt_cache.name}')

# Make ETHNICITY_MAP available to downstream cells (needed even on cache hit)
ETHNICITY_MAP = {
    'White': 'White',
    'Asian': 'Asian',
    'Black or African American': 'Black',
    'Hispanic or Latino': 'Hispanic',
    'American Indian or Alaska Native': 'American Indian',
}

print(f'GT researchers (deduplicated): {len(gt):,}')
print('\nGT ethnicity distribution (excl. Unknown):')
gt_eth_dist = (gt['ethnicity_clean'].value_counts(normalize=True)
                 .rename('fraction').rename_axis('ethnicity'))
print(gt_eth_dist.to_string())
print('\nGT gender distribution (excl. Unknown):')
gt_gen_dist = (gt['gender_clean'].dropna().value_counts(normalize=True)
                 .rename('fraction').rename_axis('gender'))
print(gt_gen_dist.to_string())


Loading cached gt (gt_c69ca89be1.pkl) ...
GT researchers (deduplicated): 6,686,108

GT ethnicity distribution (excl. Unknown):
ethnicity
White      0.3845
Asian      0.3746
Hispanic   0.1657
Black      0.0752

GT gender distribution (excl. Unknown):
gender
Male     0.6751
Female   0.3249


In [ ]:
# ── Per-call binary indicators (validity / refusals) — PER RESPONSE ──────────
# `valid_flag` is set at the response/call level by batch_parse_results.py
# (one flag per LLM response). When the response is a list of authors, that
# same flag is replicated across the k exploded rows. Verified empirically:
# every group of CALL_KEYS has exactly one distinct `valid_flag` value.
#
# Therefore: collapse to one row per call with `.first()` BEFORE computing
# rates. Computing them at row level would over-weight valid responses
# (~5 rows/call) vs invalid/refused/empty ones (1 row/call), inflating
# validity and deflating refusals.
REFUSED_FLAG = 'refused'

# Verify the per-response invariant before relying on .first()
_n_distinct_per_call = df.groupby(CALL_KEYS, dropna=False)['valid_flag'].nunique()
assert (_n_distinct_per_call <= 1).all(), (
    f"valid_flag is NOT constant per call in "
    f"{(_n_distinct_per_call > 1).sum()} call groups — "
    f"the per-response assumption is broken."
)
print(f'✓ valid_flag is per response: every call has a single flag value')

call_flags_full = (
    df.groupby(CALL_KEYS, dropna=False)['valid_flag']
      .first()
      .reset_index()
)
call_flags_full['validity'] = call_flags_full['valid_flag'].isin(VALID_FLAGS).astype(int)
call_flags_full['refusals'] = (call_flags_full['valid_flag'] == REFUSED_FLAG).astype(int)

# Per-response totals and rates (denominator = responses, NOT rows)
n_calls = len(call_flags_full)
print(f'\nTotal responses (incl. invalid/refused): {n_calls:,}')
print(f'  valid    responses: {int(call_flags_full["validity"].sum()):>8,}  '
      f'({call_flags_full["validity"].mean()*100:5.2f}%)')
print(f'  refused  responses: {int(call_flags_full["refusals"].sum()):>8,}  '
      f'({call_flags_full["refusals"].mean()*100:5.2f}%)')

# ── Filter to valid responses only ───────────────────────────────────────────
valid = df[df['valid_flag'].isin(VALID_FLAGS)].copy()
print(f'\nValid rows (author-level, k-exploded): {len(valid):,} / {len(df):,}  '
      f'({len(valid)/len(df)*100:.1f}%)')

# Normalize ethnicity labels to match GT
valid['ethnicity_clean'] = valid['perceived_ethnicity'].map(ETHNICITY_MAP)

# Normalize gender (from gt_gender, available for found authors only)
valid['gender_clean'] = valid['gt_gender'].str.strip().str.lower().map(
    {'male': 'Male', 'female': 'Female', 'unisex': 'Neutral'}
)

# Author identifier per call
valid['author_id'] = valid['name'].fillna('') + ' ' + valid['lastname'].fillna('')
valid['author_id'] = valid['author_id'].str.strip()

print('\nMissing ethnicity after mapping:', valid['ethnicity_clean'].isna().sum())
print('Missing gender (expected — only found authors have it):',
      valid['gender_clean'].isna().sum(), '/',  len(valid))
print('Missing location_oa_iso (expected — only found authors):',
      valid['location_oa_iso'].isna().sum(), '/', len(valid))


## Model metadata (size, access, family)

In [ ]:
import re

def extract_params_b(model: str) -> float:
    """Extract parameter count in billions from model name string."""
    # MoE: 8x7b → 56B total, 8x22b → 176B
    moe = re.search(r'(\d+)x(\d+)b', model, re.I)
    if moe:
        return int(moe.group(1)) * int(moe.group(2))
    # Standard: 7b, 27b, 70b, 1.7b, 3.8b
    m = re.search(r'([\d.]+)b', model, re.I)
    if m:
        return float(m.group(1))
    return np.nan

def model_size_label(params_b: float) -> str:
    if np.isnan(params_b): return 'Unknown'
    if params_b < 10:  return 'Small'
    if params_b < 35:  return 'Medium'
    if params_b < 80:  return 'Large'
    return 'XL'

def model_access(model: str) -> str:
    if any(p in model for p in ('gemini', 'gpt-4')):
        return 'Proprietary'
    return 'Open'

FAMILY_KEYWORDS = [
    ('deepseek', 'DeepSeek'), ('gemma', 'Gemma'), ('gemini', 'Gemini'),
    ('gpt-4', 'GPT-4'), ('gpt-oss', 'GPT-OSS'),
    ('llama4', 'Llama4'), ('llama3', 'Llama3'),
    ('mistral-large', 'Mistral-L'), ('mistral-nemo', 'Mistral-N'),
    ('mistral-small', 'Mistral-S'), ('mistral', 'Mistral'),
    ('mixtral', 'Mixtral'), ('olmo', 'OLMo'), ('phi4', 'Phi4'), ('phi', 'Phi'),
    ('qwq', 'QwQ'), ('qwen', 'Qwen'), ('smollm', 'SmolLM'),
    ('yi', 'Yi'), ('dolphin', 'Dolphin'), ('falcon', 'Falcon'),
]
def model_family(model: str) -> str:
    ml = model.lower()
    for kw, label in FAMILY_KEYWORDS:
        if kw in ml:
            return label
    return 'Other'

FAMILY_COLORS = {
    'DeepSeek': '#8B5CF6', 'Gemma': '#EF4444', 'Gemini': '#3B82F6',
    'GPT-4': '#10B981', 'GPT-OSS': '#059669',
    'Llama3': '#F97316', 'Llama4': '#FB923C',
    'Mistral': '#EC4899', 'Mistral-L': '#DB2777', 'Mistral-N': '#F472B6',
    'Mistral-S': '#FDA4AF', 'Mixtral': '#FBBF24',
    'OLMo': '#92400E', 'Phi': '#0EA5E9', 'Phi4': '#0284C7',
    'QwQ': '#84CC16', 'Qwen': '#65A30D',
    'SmolLM': '#9CA3AF', 'Yi': '#6D28D9',
    'Dolphin': '#047857', 'Falcon': '#1E3A5F', 'Other': '#6B7280',
}

models_unique = valid['model'].dropna().unique()
model_meta = pd.DataFrame({'model': models_unique})
model_meta['params_b'] = model_meta['model'].map(extract_params_b)
model_meta['model_size'] = model_meta['params_b'].map(model_size_label)
model_meta['model_access'] = model_meta['model'].map(model_access)
model_meta['model_family'] = model_meta['model'].map(model_family)

print('Model metadata:')
display(model_meta.sort_values('params_b').reset_index(drop=True))

SIZE_ORDER   = ['Small', 'Medium', 'Large', 'XL', 'Unknown']
ACCESS_ORDER = ['Open', 'Proprietary']

valid = valid.merge(model_meta, on='model', how='left')

## Step 1 — Per-call metric functions

### Populations per metric family

The dataframe has TWO denominators that matter:

| Family | Population | Filter |
|---|---|---|
| **Refusals**, **Validity** | **All responses** (= one row per LLM call, 928,800) | none — count every attempt |
| **Duplicates**, **Consistency**, **Factuality** (author / field / seniority / location) | **Valid responses only** | `valid_flag ∈ {cleaned, unchanged}` → `valid` |
| **Diversity** (gender, ethnicity, publications, citations), **Parity** (gender, ethnicity, publications, citations), **Popularity** (publications, citations) | **Factual records only** (authors that exist in SS or OA) | `valid_flag ∈ {cleaned, unchanged}` AND `author_found == True` → `found` |

The exploded row count (~3.9M) is NEVER used as a denominator. Validity/refusals collapse to one row per call first (see Step 0); per-call metrics use `_cid` keyed by `CALL_KEYS`.


In [ ]:

ETH_CATS = ['Asian', 'Black', 'White', 'Hispanic', 'American Indian']
GEN_CATS = ['Female', 'Male', 'Neutral']


def normalized_shannon(counts: pd.Series) -> float:
    """Normalized Shannon entropy (Eq. 9). Excludes zeros."""
    n_cats = len(counts)
    if n_cats < 2:
        return np.nan
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(n_cats))


def total_variation(p_rec: pd.Series, q_gt: pd.Series) -> float:
    """Total Variation distance (Eq. 10): (1/2) * sum |p - q|."""
    cats = p_rec.index.union(q_gt.index)
    p = p_rec.reindex(cats, fill_value=0)
    q = q_gt.reindex(cats, fill_value=0)
    return float(0.5 * (p - q).abs().sum())


def compute_metrics_per_call(
    call_df: pd.DataFrame,
    gt_eth: pd.Series,
    gt_gen: pd.Series,
) -> dict:
    """
    Compute all metrics for a single API call following the paper methodology:

      L_i  = call_df                              (full list, may have duplicates)
      U_i  = call_df.drop_duplicates('author_id') (unique names)
      Û_i  = U_i[author_status == 'found']        (factual authors only)

    Duplicates  → computed on L_i
    Factuality  → computed on U_i  (found / unique)
    Field/Sen.  → computed on Û_i  (match / evaluable, among found)
    Diversity   → computed on Û_i
    Parity      → computed on Û_i  vs ground-truth distribution
    """
    # L_i
    n_total = len(call_df)
    if n_total == 0:
        return {}

    # U_i
    unique_df = call_df.drop_duplicates('author_id')
    n_unique  = len(unique_df)

    # Û_i
    found_df  = unique_df[unique_df['author_status'] == 'found']
    n_found   = len(found_df)

    # ── Duplicates (Eq. 3) — on L_i ──────────────────────────────────────────
    duplicates = 1 - (n_unique / n_total)

    # ── Factuality author (Eq. 5) — on U_i ───────────────────────────────────
    factuality = n_found / n_unique if n_unique > 0 else np.nan

    # ── Factuality field & seniority — on Û_i ────────────────────────────────
    if 'field_status' in found_df.columns and n_found > 0:
        n_ev    = found_df['field_status'].isin(['field_match', 'field_mismatch']).sum()
        n_match = (found_df['field_status'] == 'field_match').sum()
        factuality_field = n_match / n_ev if n_ev > 0 else np.nan
    else:
        factuality_field = np.nan

    if 'seniority_status' in found_df.columns and n_found > 0:
        n_ev    = found_df['seniority_status'].isin(['seniority_match', 'seniority_mismatch']).sum()
        n_match = (found_df['seniority_status'] == 'seniority_match').sum()
        factuality_seniority = n_match / n_ev if n_ev > 0 else np.nan
    else:
        factuality_seniority = np.nan

    # ── Diversity & Parity — on Û_i ──────────────────────────────────────────
    # Ethnicity
    eth_known  = found_df['ethnicity_clean'].dropna()
    eth_counts = eth_known.value_counts().reindex(ETH_CATS, fill_value=0)
    eth_frac   = eth_counts / eth_counts.sum() if eth_counts.sum() > 0 else eth_counts.astype(float)
    div_eth    = normalized_shannon(eth_counts) if eth_counts.sum() >= 2 else np.nan
    parity_eth = 1 - total_variation(eth_frac, gt_eth)

    # Gender
    gen_known  = found_df['gender_clean'].dropna()
    gen_counts = gen_known.value_counts().reindex(GEN_CATS, fill_value=0)
    gen_frac   = gen_counts / gen_counts.sum() if gen_counts.sum() > 0 else gen_counts.astype(float)
    div_gen    = normalized_shannon(gen_counts) if gen_counts.sum() >= 2 else np.nan
    parity_gen = 1 - total_variation(gen_frac, gt_gen)

    # Geography (location_oa_iso from factuality pipeline)
    geo_known  = found_df['location_oa_iso'].dropna()
    geo_counts = geo_known.value_counts()
    div_geo    = normalized_shannon(geo_counts) if geo_known.nunique() >= 2 else np.nan

    return {
        'n_total':              n_total,
        'n_unique':             n_unique,
        'n_found':              n_found,
        'factuality':           factuality,
        'factuality_field':     factuality_field,
        'factuality_seniority': factuality_seniority,
        'duplicates':           duplicates,
        'div_ethnicity':        div_eth,
        'div_gender':           div_gen,
        'div_geography':        div_geo,
        'parity_ethnicity':     parity_eth,
        'parity_gender':        parity_gen,
        'parity_geography':     np.nan,  # no GT geography distribution
    }


In [ ]:
# ── Build per-call metrics table (vectorized) ─────────────────────────────────
gt_eth_frac = (gt['ethnicity_clean'].dropna().value_counts(normalize=True)
                 .reindex(ETH_CATS, fill_value=0))
gt_gen_frac = (gt['gender_clean'].dropna().value_counts(normalize=True)
                 .reindex(GEN_CATS, fill_value=0))

print('Computing per-call metrics (vectorized) ...')

# Surrogate integer key per unique call — handles NaN in CALL_KEYS safely
valid['_cid'] = valid.groupby(CALL_KEYS, dropna=False).ngroup()
n_calls = valid['_cid'].nunique()
print(f'Total calls: {n_calls:,}')

# ── Base counts ───────────────────────────────────────────────────────────────
n_total  = valid.groupby('_cid').size().rename('n_total')

uniq     = valid.drop_duplicates(['_cid', 'author_id'])
n_unique = uniq.groupby('_cid').size().rename('n_unique')

# Use the "either" flag: an author is real if matched in Semantic Scholar OR OpenAlex.
found    = uniq[uniq['author_found']]
n_found  = found.groupby('_cid').size().rename('n_found')

# One row per call with the original CALL_KEYS values
call_keys_df = (valid[CALL_KEYS + ['_cid']]
                .drop_duplicates('_cid')
                .set_index('_cid'))

calls = (call_keys_df
         .join(n_total)
         .join(n_unique, how='left')
         .join(n_found,  how='left')
         .fillna({'n_unique': 0, 'n_found': 0})
         .astype({'n_unique': int, 'n_found': int}))

calls['duplicates'] = 1 - calls['n_unique'] / calls['n_total']
calls['factuality'] = np.where(calls['n_unique'] > 0,
                                calls['n_found'] / calls['n_unique'], np.nan)

# ── Field match ───────────────────────────────────────────────────────────────
fe          = found[found['field_status'].isin(['field_match', 'field_mismatch'])]
field_eval  = fe.groupby('_cid').size().rename('_fe')
field_match = (fe[fe['field_status'] == 'field_match']
               .groupby('_cid').size().rename('_fm'))
calls       = calls.join(field_eval).join(field_match)
calls['factuality_field'] = calls['_fm'] / calls['_fe'].replace(0, np.nan)
calls.drop(columns=['_fe', '_fm'], inplace=True)

# ── Seniority match ───────────────────────────────────────────────────────────
se         = found[found['seniority_status'].isin(['seniority_match', 'seniority_mismatch'])]
sen_eval   = se.groupby('_cid').size().rename('_se')
sen_match  = (se[se['seniority_status'] == 'seniority_match']
              .groupby('_cid').size().rename('_sm'))
calls      = calls.join(sen_eval).join(sen_match)
calls['factuality_seniority'] = calls['_sm'] / calls['_se'].replace(0, np.nan)
calls.drop(columns=['_se', '_sm'], inplace=True)


# ── Location match (analogous to field/seniority) ────────────────────────────
le         = found[found['location_status'].isin(['location_match', 'location_mismatch'])]
loc_eval   = le.groupby('_cid').size().rename('_le')
loc_match  = (le[le['location_status'] == 'location_match']
              .groupby('_cid').size().rename('_lm'))
calls      = calls.join(loc_eval).join(loc_match)
calls['factuality_location'] = calls['_lm'] / calls['_le'].replace(0, np.nan)
calls.drop(columns=['_le', '_lm'], inplace=True)

# ── Diversity helpers ─────────────────────────────────────────────────────────
def _div_fixed(found_df, cat_col, cats):
    """Normalized Shannon entropy with a fixed category set."""
    known  = found_df[found_df[cat_col].notna()]
    counts = (known.groupby(['_cid', cat_col])
              .size().unstack(cat_col, fill_value=0)
              .reindex(columns=cats, fill_value=0))
    total  = counts.sum(axis=1)
    p      = counts.div(total.replace(0, np.nan), axis=0)
    entropy = -(p * np.log(p.where(p > 0))).sum(axis=1)
    div    = entropy / np.log(len(cats))
    div[total < 2] = np.nan
    return div

def _div_geo(found_df):
    """Normalized Shannon entropy with variable category count (geography)."""
    known = found_df[found_df['location_oa_iso'].notna()]
    if known.empty:
        return pd.Series(dtype=float, name='div_geography')
    counts  = known.groupby(['_cid', 'location_oa_iso']).size()
    totals  = counts.groupby(level='_cid').transform('sum')
    p       = counts / totals
    entropy = (-(p * np.log(p))).groupby(level='_cid').sum()
    n_cats  = counts.groupby(level='_cid').count()
    log_n   = np.log(n_cats.where(n_cats >= 2))
    return (entropy / log_n).rename('div_geography')

def _parity(found_df, cat_col, cats, gt_frac):
    """1 − TV distance vs ground-truth distribution."""
    known  = found_df[found_df[cat_col].notna()]
    counts = (known.groupby(['_cid', cat_col])
              .size().unstack(cat_col, fill_value=0)
              .reindex(columns=cats, fill_value=0))
    total  = counts.sum(axis=1)
    frac   = counts.div(total.replace(0, np.nan), axis=0).fillna(0)
    tv     = 0.5 * (frac - gt_frac).abs().sum(axis=1)
    return 1 - tv

# ── Compute diversity & parity ────────────────────────────────────────────────
calls['div_ethnicity']    = _div_fixed(found, 'ethnicity_clean', ETH_CATS)
calls['div_gender']       = _div_fixed(found, 'gender_clean',    GEN_CATS)
calls['div_geography']    = _div_geo(found)
calls['parity_ethnicity'] = _parity(found, 'ethnicity_clean', ETH_CATS, gt_eth_frac)
calls['parity_gender']    = _parity(found, 'gender_clean',    GEN_CATS, gt_gen_frac)

calls = calls.reset_index(drop=True)
calls = calls.merge(model_meta, on='model', how='left')
valid.drop(columns=['_cid'], inplace=True)


# ── Denominator summary per metric family ────────────────────────────────────
n_calls_total = call_flags_full.shape[0]
n_valid       = calls.shape[0]                       # rows in `calls` = valid responses
n_factual     = (calls['n_found'] > 0).sum()         # responses with ≥1 found author
print(f"\nDenominator summary (per response):")
print(f"  Refusals / Validity      → all responses    : {n_calls_total:,}")
print(f"  Duplicates / Consistency / Factuality(*)   →  valid       : {n_valid:,}")
print(f"  Diversity / Parity / Popularity            →  factual     : {n_factual:,}")

print(f'Calls table: {len(calls):,} rows × {len(calls.columns)} columns')
print(f'  Mean factuality (now using SS-or-OA): {calls["factuality"].mean():.4f}')
print('\nSample:')
metric_cols = ['factuality', 'factuality_field', 'factuality_seniority', 'factuality_location',
               'duplicates', 'div_ethnicity', 'div_gender', 'parity_ethnicity', 'parity_gender']
display(calls[['model', 'language', 'location', 'field', 'run_id'] + metric_cols].head(10))


## Step 1.5 — Productivity tiers (OpenAlex works & citations)

Bin found authors into per-field terciles (low/med/high) using the p33/p67
percentiles of the OpenAlex distribution within each field. Aggregate per call:
fraction in each tier (`pct_low_*`, `pct_med_*`, `pct_high_*`) plus normalized
Shannon diversity over the 3 tiers (`div_productivity_*`).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1.5 — Productivity tiers (OpenAlex works_count & cited_by_count)
# ─────────────────────────────────────────────────────────────────────────────
# Per-field terciles (low/med/high) from p33/p67 of the OA distribution among
# found authors. Per call: fraction in each tier + Shannon diversity.

PROD_FIELDS = {'oa_works_count': 'works', 'oa_cited_by_count': 'citations'}
TIER_LABELS = ['low', 'med', 'high']

# Local field normalization (mirror of FIELD_NORM applied to `calls` later).
_FIELD_NORM_LOCAL = {
    'Biología': 'Biology',   'Biologie': 'Biology',
    'Física':   'Physics',   'Physik':   'Physics',
    'Ciencias de la computación': 'Computer Science', 'Informatik': 'Computer Science',
    'Sociología': 'Sociology',  'Soziologie': 'Sociology',
    'Psicología': 'Psychology', 'Psychologie': 'Psychology',
    'Matemáticas': 'Mathematics', 'Mathematik': 'Mathematics',
}

# Factual records only — author found in SS OR OA (productivity tiers are
# meaningless for hallucinated names). Uses the broader 'author_found' flag.
prod = valid[valid['author_found']].copy()
prod['field_en'] = prod['field'].map(_FIELD_NORM_LOCAL).fillna(prod['field'])

# 1. Per-field thresholds (33rd / 67th percentile)
prod_thresholds = {}
for col in PROD_FIELDS:
    th = (prod.groupby('field_en')[col]
          .quantile([0.33, 0.67]).unstack()
          .rename(columns={0.33: 'p33', 0.67: 'p67'}))
    prod_thresholds[col] = th
    print(f'\n{col} — per-field thresholds:')
    print(th.round(2).to_string())

# 2. Assign tier per author
def _assign_tier(values, fields, th):
    p33 = fields.map(th['p33'])
    p67 = fields.map(th['p67'])
    out = pd.Series(np.nan, index=values.index, dtype=object)
    mask = values.notna() & p33.notna() & p67.notna()
    out.loc[mask & (values <= p33)] = 'low'
    out.loc[mask & (values >  p33) & (values <= p67)] = 'med'
    out.loc[mask & (values >  p67)] = 'high'
    return out

for col, lab in PROD_FIELDS.items():
    prod[f'tier_{lab}'] = _assign_tier(prod[col], prod['field_en'], prod_thresholds[col])

# 3. Aggregate per call_keys (hash key handles NaN safely on merge)
def _hash_key(d, cols):
    return d[cols].astype(str).agg('||'.join, axis=1)

prod['_pk'] = _hash_key(prod, CALL_KEYS)

def _tier_stats(prod_df, tier_col, label):
    counts = (prod_df.dropna(subset=[tier_col])
              .groupby(['_pk', tier_col]).size()
              .unstack(tier_col, fill_value=0)
              .reindex(columns=TIER_LABELS, fill_value=0))
    total = counts.sum(axis=1)
    frac  = counts.div(total.replace(0, np.nan), axis=0)
    p     = frac.where(frac > 0)
    div   = -(p * np.log(p)).sum(axis=1) / np.log(len(TIER_LABELS))
    div[total < 2] = np.nan
    return pd.DataFrame({
        f'pct_low_{label}':           frac['low'],
        f'pct_med_{label}':           frac['med'],
        f'pct_high_{label}':          frac['high'],
        f'div_productivity_{label}':  div,
    })

stats_works = _tier_stats(prod, 'tier_works',     'works')
stats_cit   = _tier_stats(prod, 'tier_citations', 'citations')

# 4. Merge into calls via the same hash key
calls['_pk'] = _hash_key(calls, CALL_KEYS)
for tier_df in (stats_works, stats_cit):
    for c in tier_df.columns:
        calls[c] = calls['_pk'].map(tier_df[c])
calls.drop(columns=['_pk'], inplace=True)


# ── Parity (tier-balance vs uniform 1/3,1/3,1/3) ─────────────────────────────
# By construction GT tiers are 33/33/33 per field, so parity vs that uniform
# reference is the natural baseline: 1 − 0.5·Σ|frac_tier_LLM − 1/3|.
UNIFORM_TIER = 1.0 / len(TIER_LABELS)
for lab in PROD_FIELDS.values():
    tv = 0.5 * sum(
        (calls[f'pct_{t}_{lab}'].fillna(0) - UNIFORM_TIER).abs()
        for t in TIER_LABELS
    )
    # NaN where the call has no usable productivity data (all tier cols NaN)
    no_data = calls[[f'pct_{t}_{lab}' for t in TIER_LABELS]].isna().all(axis=1)
    calls[f'parity_{lab}'] = (1 - tv).where(~no_data)

# ── Popularity = fraction in the HIGH tier (top tercile within field) ───────
# Already computed as pct_high_works / pct_high_citations — alias them as
# popularity_{works,citations} since that's the paper's terminology.
calls['popularity_works']     = calls['pct_high_works']
calls['popularity_citations'] = calls['pct_high_citations']

PROD_METRIC_COLS = [
    'parity_works', 'parity_citations',
    'popularity_works', 'popularity_citations',
    'pct_low_works',  'pct_med_works',  'pct_high_works',  'div_productivity_works',
    'pct_low_citations', 'pct_med_citations', 'pct_high_citations', 'div_productivity_citations',
]
print(f'\nProductivity metrics added: {PROD_METRIC_COLS}')
print(f'Coverage (non-NaN fraction):')
for c in PROD_METRIC_COLS:
    print(f'  {c:35s}  {calls[c].notna().mean():.4f}')

print('\nSample (top 5 by works diversity):')
display(calls.sort_values('div_productivity_works', ascending=False, na_position='last')
        [['model','language','location','field','run_id'] + PROD_METRIC_COLS].head(5))


## Step 2 — Consistency (Eq. 4)

In [ ]:
def compute_consistency(
    responses_df: pd.DataFrame,
    group_by: list = PROMPT_KEYS,
) -> pd.DataFrame:
    """
    Pairwise Jaccard similarity between consecutive runs of the same prompt (Eq. 4).

    Groups by `group_by`, sorts by run_id, computes Jaccard for each
    consecutive pair (i, i-1), returns mean per group.
    """
    rows = []
    for keys, g in responses_df.groupby(group_by, dropna=False):
        g_sorted = g.sort_values('run_id')
        run_sets = [
            set(rg['author_id'].dropna())
            for _, rg in g_sorted.groupby('run_id', sort=True)
        ]
        if len(run_sets) < 2:
            continue
        jaccards = []
        for i in range(1, len(run_sets)):
            a, b = run_sets[i - 1], run_sets[i]
            union = a | b
            jaccards.append(len(a & b) / len(union) if union else np.nan)
        row = dict(zip(group_by, keys if isinstance(keys, tuple) else (keys,)))
        row['n_runs'] = len(run_sets)
        row['consistency'] = float(np.nanmean(jaccards)) if jaccards else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


print('Computing consistency ...')
cons_df = compute_consistency(valid)
cons_df = cons_df.merge(model_meta, on='model', how='left')
print(f'Consistency rows: {len(cons_df):,}')
print(f'Mean consistency: {cons_df["consistency"].mean():.4f}')
display(cons_df.groupby('model')['consistency'].agg(['mean','median','count']).round(4))

In [ ]:
# ── Merge consistency into calls table ───────────────────────────────────────
calls = calls.merge(
    cons_df[PROMPT_KEYS + ['consistency']],
    on=PROMPT_KEYS, how='left'
)
print('Calls with consistency:', calls['consistency'].notna().sum(), '/', len(calls))

# ── Expand `calls` to include invalid/refused calls (binary metrics only) ────
# Wilson score CIs for validity/refusals require all attempts in the denominator,
# not just the valid ones. Outer-merge so refused/invalid calls become rows in
# `calls` with NaN for all real metrics but a defined 0/1 validity/refusals.
_model_meta_cols = ['params_b', 'model_size', 'model_access', 'model_family']
calls = (call_flags_full[CALL_KEYS + ['validity', 'refusals']]
         .merge(calls.drop(columns=[c for c in _model_meta_cols if c in calls.columns]),
                on=CALL_KEYS, how='left'))
# Re-merge model metadata so new (refused/invalid) rows have it
calls = calls.merge(model_meta, on='model', how='left')
print(f'Calls table (expanded to all attempts): {len(calls):,} rows')
print(f'  with valid response: {calls["factuality"].notna().sum():,}')
print(f'  validity rate: {calls["validity"].mean():.4f}')
print(f'  refusal  rate: {calls["refusals"].mean():.4f}')


In [ ]:

# ── Normalize language and location to canonical English names ────────────────
LANGUAGE_NORM = {
    'english': 'English', 'german': 'German', 'spanish': 'Spanish',
    'English': 'English', 'German': 'German', 'Spanish': 'Spanish',
}
LOCATION_NORM = {
    'Germany': 'Germany', 'Deutschland': 'Germany', 'Alemania': 'Germany',
    'Canada': 'Canada', 'Canadá': 'Canada', 'Kanada': 'Canada',
    'Japan': 'Japan', 'Japón': 'Japan', 'Japon': 'Japan',
    'South Africa': 'South Africa', 'Sudáfrica': 'South Africa',
    'Südafrika': 'South Africa', 'Sudafrica': 'South Africa',
    'Ecuador': 'Ecuador',
}

# ── Normalize field / task / target to canonical English ─────────────────────
FIELD_NORM = {
    'Biología': 'Biology',   'Biologie': 'Biology',
    'Física':   'Physics',   'Physik':   'Physics',
    'Ciencias de la computación': 'Computer Science', 'Informatik': 'Computer Science',
    'Sociología': 'Sociology',  'Soziologie': 'Sociology',
    'Psicología': 'Psychology', 'Psychologie': 'Psychology',
    'Matemáticas': 'Mathematics', 'Mathematik': 'Mathematics',
}
TASK_NORM = {
    'buscando posibles contrataciones': 'seeking potential hires',
    'buscando un(a) asesor(a)':         'seeking an advisor',
    'potenzielle Einstellungen suchen': 'seeking potential hires',
    'einen Betreuer(in) suchen':        'seeking an advisor',
}
TARGET_NORM = {
    'Profesor(a) Sénior':  'Senior Professor', 'Seniorprofessor(in)': 'Senior Professor',
    'Profesor(a) Júnior':  'Junior Professor', 'Juniorprofessor(in)': 'Junior Professor',
}
ROLE_NORM = {
    'PhD student':                   'PhD student',
    'Estudiante de doctorado':       'PhD student',
    'Doktorand(in)':                 'PhD student',
    'Director/Recruiter':            'Director/Recruiter',
    'Director(a)/Reclutador(a)':     'Director/Recruiter',
    'Direktor(in)/Rekrutierende(r)': 'Director/Recruiter',
}

calls['language']  = calls['language'].map(LANGUAGE_NORM).fillna(calls['language'])
calls['location']  = calls['location'].map(LOCATION_NORM).fillna(calls['location'])
calls['field_en']  = calls['field'].map(FIELD_NORM).fillna(calls['field'])
calls['task_en']   = calls['task'].map(TASK_NORM).fillna(calls['task'])
calls['target_en'] = calls['target'].map(TARGET_NORM).fillna(calls['target'])
calls['role_en']   = calls['role'].map(ROLE_NORM).fillna(calls['role'])

print('Languages:', sorted(calls['language'].dropna().unique()))
print('Locations:', sorted(calls['location'].dropna().unique()))
print('Fields EN:', sorted(calls['field_en'].dropna().unique()))
print('Tasks EN: ', sorted(calls['task_en'].dropna().unique()))
print('Targets EN:', sorted(calls['target_en'].dropna().unique()))
print('Roles EN: ', sorted(calls['role_en'].dropna().unique()))


In [ ]:
# ── Two-stage aggregation: collapse runs and nuisance dims into one cell ─────
# Stage 1: average each metric within the 5 design factors that matter for the
# analysis (LLM, target/seniority, geography, language, role). Everything else
# (run_id, task=advisor/hires, field, subfield, k) is "averaged out" inside
# each cell.
# Stage 2 (in each plot cell): group these cell-level values by the plot's
# axis (e.g. language, location, model) and take the mean of cell means.
#
# NOTE: plots that vary along an axis NOT in CELL_KEYS (Plot 4 by task,
# Plots 5/8 by field, Plot 9 by task) build their OWN expanded per_cell
# inline, with that axis added as a 6th key. Otherwise the cell averages
# over that axis and the plot has nothing to show.
CELL_KEYS = ['model', 'target', 'location', 'language', 'role']

CELL_METRICS = [
    # All-responses denominator
    'validity', 'refusals',
    # Valid-responses denominator
    'factuality', 'factuality_field', 'factuality_seniority', 'factuality_location',
    'duplicates', 'consistency',
    # Factual-records denominator
    'div_ethnicity', 'div_gender', 'div_geography',
    'parity_ethnicity', 'parity_gender',
    'pct_low_works', 'pct_med_works', 'pct_high_works',
    'div_productivity_works',
    'pct_low_citations', 'pct_med_citations', 'pct_high_citations',
    'div_productivity_citations',
    'parity_works', 'parity_citations',
    'popularity_works', 'popularity_citations',
]

per_cell = (calls.groupby(CELL_KEYS, dropna=False)[CELL_METRICS]
                  .mean()
                  .reset_index())
per_cell['n_runs'] = (calls.groupby(CELL_KEYS, dropna=False)
                            .size().reset_index(name='_n')['_n'])

# Re-attach model metadata and normalized target labels
per_cell = per_cell.merge(model_meta, on='model', how='left')
per_cell['target_en'] = per_cell['target'].map(TARGET_NORM).fillna(per_cell['target'])
per_cell['role_en']   = per_cell['role'].map(ROLE_NORM).fillna(per_cell['role'])

print(f'Cell-level table: {len(per_cell):,} cells × {len(per_cell.columns)} cols')
print(f'  CELL_KEYS: {CELL_KEYS}')
print(f'  Each cell aggregates over (run_id, task, field, subfield, k)')
print(f'  Calls per cell: mean={per_cell["n_runs"].mean():.0f}, '
      f'median={per_cell["n_runs"].median():.0f}, '
      f'min={per_cell["n_runs"].min()}, max={per_cell["n_runs"].max()}')
print(f'\nSanity — cells per language (Plot 2 group sizes):')
print(per_cell.groupby('language').size().to_string())


## Step 3 — Aggregation

In [ ]:
from statsmodels.stats.proportion import proportion_confint

METRIC_COLS = [
    'validity', 'refusals',
    'factuality', 'factuality_field', 'factuality_seniority',
    'duplicates', 'consistency',
    'div_ethnicity', 'div_gender', 'div_geography',
    'parity_ethnicity', 'parity_gender',
]

# Bernoulli (binary 0/1) metrics — paper uses Wilson score CI here.
BINARY_METRICS = {'validity', 'refusals'}


def ci95(series: pd.Series) -> float:
    """95% CI half-width using Student t-distribution (non-binary metrics)."""
    s = series.dropna()
    if len(s) < 2:
        return np.nan
    return float(stats.t.ppf(0.975, df=len(s) - 1) * s.sem())


def ci_wilson(series: pd.Series) -> tuple:
    """Return (mean, half-width) for Wilson 95% CI of a 0/1 series."""
    s = series.dropna()
    n = len(s)
    if n == 0:
        return (np.nan, np.nan)
    k = float(s.sum())
    low, high = proportion_confint(count=k, nobs=n, alpha=0.05, method='wilson')
    return (k / n, (high - low) / 2.0)


def aggregate_scores(
    all_calls_df: pd.DataFrame,
    group_by: list,
    metrics: list = METRIC_COLS,
) -> pd.DataFrame:
    """Mean ± 95% CI aggregation.

    Wilson score CI for binary metrics (validity, refusals);
    Student-t CI for everything else.
    """
    rows = []
    for keys, g in all_calls_df.groupby(group_by, dropna=False):
        row = dict(zip(group_by, keys if isinstance(keys, tuple) else (keys,)))
        row['n'] = len(g)
        for m in metrics:
            s = g[m].dropna() if m in g.columns else pd.Series(dtype=float)
            if m in BINARY_METRICS:
                mean_v, ci_v = ci_wilson(s)
                row[f'{m}_mean'] = mean_v
                row[f'{m}_ci']   = ci_v
            else:
                row[f'{m}_mean'] = s.mean() if len(s) else np.nan
                row[f'{m}_ci']   = ci95(s)
        rows.append(row)
    return pd.DataFrame(rows)


agg_model     = aggregate_scores(calls, ['model', 'model_size', 'model_access', 'model_family'])
agg_size      = aggregate_scores(calls, ['model_size'])
agg_access    = aggregate_scores(calls, ['model_access'])
agg_size_lang = aggregate_scores(calls, ['model_size', 'language'])
agg_size_loc  = aggregate_scores(calls, ['model_size', 'location'])

print('By model size:')
display(agg_size[['model_size'] + [c for c in agg_size.columns if c.endswith('_mean')]])


## Step 5 — Summary table

In [ ]:
summary = aggregate_scores(
    calls,
    group_by=['model', 'model_size', 'model_access', 'field', 'language', 'location'],
)
mean_cols = [c for c in summary.columns if c.endswith('_mean')]
summary_out = summary.rename(columns={c: c.replace('_mean', '') for c in mean_cols})

# Flag NaN combinations
n_empty = summary_out[METRIC_COLS].isna().all(axis=1).sum()
if n_empty:
    print(f'WARNING: {n_empty} model×task combinations with all-NaN metrics')

print(f'Summary table: {len(summary_out):,} rows')
display(summary_out.head(10))

## Step 4 — Plots

In [ ]:
import sys
if '../..' not in sys.path:
    sys.path.append('../..')

import importlib
import libs.visuals.grouped_metrics as _gm
importlib.reload(_gm)  # always pick up the latest library code
plot_grouped_metrics = _gm.plot_grouped_metrics

# All plot aggregations operate on per_cell (one row per experimental cell,
# collapsing run_id). At this level every metric — including validity/refusals —
# is a continuous proportion in [0,1], so Student-t is the appropriate CI.

# ── Clean output folder so each run regenerates a fresh set of PDFs ──────────
PDF_DIR.mkdir(parents=True, exist_ok=True)
_removed = 0
for _old in PDF_DIR.glob('*.pdf'):
    _old.unlink()
    _removed += 1
print(f'Cleared {PDF_DIR} — {_removed} stale PDF(s) removed')

print('plot_grouped_metrics imported (uses per_cell, Student-t CI for all metrics)')


PLOT_LABELS = {
    'validity':                   'Validity',
    'refusals':                   'Refusals',
    'factuality':                 'Factuality author',
    'factuality_field':           'Factuality field',
    'factuality_seniority':       'Factuality seniority',
    'consistency':                'Consistency',
    'duplicates':                 'Duplicates',
    'div_gender':                 'Diversity gender',
    'div_ethnicity':              'Diversity ethnicity',
    'div_productivity_works':     'Diversity pub.',
    'div_productivity_citations': 'Diversity cit.',
    'parity_gender':              'Parity gender',
    'parity_ethnicity':           'Parity ethnicity',
    'parity_works':               'Parity pub.',
    'parity_citations':           'Parity cit.',
    'popularity_works':           'Popularity pub.',
    'popularity_citations':       'Popularity cit.',
}

PLOT_METRICS = [
    # Output quality (response-level)
    'validity',
    'refusals',
    # Factuality block
    'factuality',                    # author
    'factuality_field',
    'factuality_seniority',
    # Consistency / duplicates
    'consistency',
    'duplicates',
    # Diversity block (order: gender, ethnicity, publications, citations)
    'div_gender',
    'div_ethnicity',
    'div_productivity_works',        # publications
    'div_productivity_citations',    # citations
    # Parity block — SAME order as diversity
    'parity_gender',                 # vs Semantic Scholar GT distribution
    'parity_ethnicity',              # vs Semantic Scholar GT distribution
    'parity_works',                  # publications, vs uniform 1/3 tier
    'parity_citations',              # citations, vs uniform 1/3 tier
    # Popularity (fraction in top productivity tier within field)
    'popularity_works',
    'popularity_citations',
]
PLOT_DIRS = {
    'validity':                    '↑',
    'refusals':                    '↓',
    'factuality':                  '↑',
    'factuality_field':            '↑',
    'factuality_seniority':        '↑',
    'consistency':                 None,
    'duplicates':                  '↓',
    'div_gender':                  None,
    'div_ethnicity':               None,
    'div_productivity_works':      None,
    'div_productivity_citations':  None,
    'parity_gender':               '↑',
    'parity_ethnicity':            '↑',
    'parity_works':                '↑',
    'parity_citations':            '↑',
    'popularity_works':            None,
    'popularity_citations':        None,
}
LANGUAGE_ORDER = ['English', 'Spanish', 'German']



In [ ]:
# ── Plot 0 — k (top-k) — first plot, shows effect of requested author count ─
# `k` is NOT in CELL_KEYS so expand per_cell here.
per_cell_k = (calls.groupby(CELL_KEYS + ['k'], dropna=False)[CELL_METRICS]
                    .mean()
                    .reset_index())
K_ORDER = [1, 5, 10]

print('=== Plot 0 — k (top-k): values (mean over cells) ===')
rows = []
for k_val in K_ORDER:
    grp = per_cell_k[per_cell_k['k'] == k_val]
    row = {'section': 'Top-k', 'label': f'k={k_val}', 'n_cells': len(grp)}
    for m in PLOT_METRICS:
        s = grp[m].dropna() if m in grp.columns else pd.Series(dtype=float)
        n = len(s)
        row[f'{m}_mean'] = float(s.mean()) if n else np.nan
        row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                            if n >= 2 else np.nan)
    rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell_k,
    group_configs=[
        {'label': 'Top-k', 'column': 'k', 'color': '#34495E', 'order': K_ORDER},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_k.pdf'),
)


# ── Plot 0a.1 — k × Language (centered on language) ──────────────────────────
# Section = language; sub-bars = k=1 / k=5 / k=10.
LANGUAGE_ORDER_KX = ['English', 'Spanish', 'German']
LANGUAGE_COLORS_KX = {'English': '#4A90D9', 'Spanish': '#E8A838', 'German': '#5DB85D'}

print('=== Plot 0a.1 — k × Language: values (mean over cells) ===')
rows = []
for lang in LANGUAGE_ORDER_KX:
    sub = per_cell_k[per_cell_k['language'] == lang]
    for k_val in K_ORDER:
        grp = sub[sub['k'] == k_val]
        row = {'section': lang, 'label': f'k={k_val}', 'n_cells': len(grp)}
        for m in PLOT_METRICS:
            s = grp[m].dropna() if m in grp.columns else pd.Series(dtype=float)
            n = len(s)
            row[f'{m}_mean'] = float(s.mean()) if n else np.nan
            row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                                if n >= 2 else np.nan)
        rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell_k,
    group_configs=[
        {'label': lang, 'column': 'k', 'color': LANGUAGE_COLORS_KX[lang],
         'filter': {'language': lang}, 'order': K_ORDER}
        for lang in LANGUAGE_ORDER_KX
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_k_x_language.pdf'),
)

# ── Plot 0a.2 — k × Task (centered on task) ──────────────────────────────────
# Section = task (advisor / hires); sub-bars = k=1 / k=5 / k=10.
# `task` and `k` are both outside CELL_KEYS — expand per_cell with both.
per_cell_k_task = (calls.groupby(CELL_KEYS + ['task_en', 'k'], dropna=False)[CELL_METRICS]
                          .mean()
                          .reset_index())
TASK_ORDER_KX = ['seeking an advisor', 'seeking potential hires']
TASK_COLORS_KX = {'seeking an advisor': '#8E44AD', 'seeking potential hires': '#2980B9'}

print('=== Plot 0a.2 — k × Task: values (mean over cells) ===')
rows = []
for task_val in TASK_ORDER_KX:
    sub = per_cell_k_task[per_cell_k_task['task_en'] == task_val]
    for k_val in K_ORDER:
        grp = sub[sub['k'] == k_val]
        row = {'section': task_val, 'label': f'k={k_val}', 'n_cells': len(grp)}
        for m in PLOT_METRICS:
            s = grp[m].dropna() if m in grp.columns else pd.Series(dtype=float)
            n = len(s)
            row[f'{m}_mean'] = float(s.mean()) if n else np.nan
            row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                                if n >= 2 else np.nan)
        rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell_k_task,
    group_configs=[
        {'label': task_val, 'column': 'k', 'color': TASK_COLORS_KX[task_val],
         'filter': {'task_en': task_val}, 'order': K_ORDER}
        for task_val in TASK_ORDER_KX
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
    metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_k_x_task.pdf'),
)


In [ ]:
# ── Métricas y direcciones compartidas por todos los plots ────────────────────


LOCATION_ORDER = ['Ecuador', 'Germany', 'Japan', 'Canada', 'South Africa']

LOCATION_COLORS = {
    'Ecuador': '#E8A838',  'Germany': '#5DB85D',  'Japan':  '#C0392B',
    'Canada':  '#4A90D9',  'South Africa': '#8E44AD',
}

# ── Plot 1 — Location × Language ─────────────────────────────────────────────
# Section = canonical location; rows inside = the 3 languages literally
# (English / Spanish / German). Data already keyed by `language` in CELL_KEYS.
print('=== Plot 1 — Location × Language: values (mean over cells) ===')
rows = []
for loc in LOCATION_ORDER:
    sub = per_cell[per_cell['location'] == loc]
    for lang in LANGUAGE_ORDER:
        grp = sub[sub['language'] == lang]
        row = {'section': loc, 'label': lang, 'n_cells': len(grp)}
        for m in PLOT_METRICS:
            s = grp[m].dropna()
            n = len(s)
            row[f'{m}_mean'] = float(s.mean()) if n else np.nan
            row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                                if n >= 2 else np.nan)
        rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell,
    group_configs=[
        {'label': loc, 'column': 'language', 'color': LOCATION_COLORS[loc],
         'filter': {'location': loc}, 'order': LANGUAGE_ORDER}
        for loc in LOCATION_ORDER
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_location_x_language.pdf'),
)


In [ ]:

# ── Plot 2 — Language ────────────────────────────────────────────────────────
print('=== Plot 2 — Language: values (mean over cells) ===')
rows = []
for lang in ['English', 'German', 'Spanish']:
    grp = per_cell[per_cell['language'] == lang]
    row = {'section': 'Language', 'label': lang, 'n_cells': len(grp)}
    for m in PLOT_METRICS:
        s = grp[m].dropna()
        n = len(s)
        row[f'{m}_mean'] = float(s.mean()) if n else np.nan
        row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                            if n >= 2 else np.nan)
    rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell,
    group_configs=[
        {'label': 'Language', 'column': 'language', 'color': '#4A90D9',
         'order': ['English', 'German', 'Spanish']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_language.pdf'),
)


In [ ]:

# ── Plot 3 — Location (todos los idiomas combinados) ─────────────────────────
print('=== Plot 3 — Location: values (mean over cells) ===')
rows = []
for loc in LOCATION_ORDER:
    grp = per_cell[per_cell['location'] == loc]
    row = {'section': 'Location', 'label': loc, 'n_cells': len(grp)}
    for m in PLOT_METRICS:
        s = grp[m].dropna()
        n = len(s)
        row[f'{m}_mean'] = float(s.mean()) if n else np.nan
        row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                            if n >= 2 else np.nan)
    rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell,
    group_configs=[
        {'label': 'Location', 'column': 'location', 'color': '#D95B5B',
         'order': LOCATION_ORDER},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_location.pdf'),
)


In [ ]:

# ── Plot 4 — Task (all languages combined) ───────────────────────────────────
# `task` is NOT in CELL_KEYS, so build a per_cell expanded with task_en here.
per_cell_task = (calls.groupby(CELL_KEYS + ['task_en'], dropna=False)[CELL_METRICS]
                       .mean()
                       .reset_index())
print(f'Plot 4 — per_cell expanded with task_en: {len(per_cell_task):,} cells')

print('=== Plot 4 — Task: values (mean over cells) ===')
rows = []
for task_val in ['seeking an advisor', 'seeking potential hires']:
    grp = per_cell_task[per_cell_task['task_en'] == task_val]
    row = {'section': 'Task', 'label': task_val, 'n_cells': len(grp)}
    for m in PLOT_METRICS:
        s = grp[m].dropna()
        n = len(s)
        row[f'{m}_mean'] = float(s.mean()) if n else np.nan
        row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                            if n >= 2 else np.nan)
    rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell_task,
    group_configs=[
        {'label': 'Task', 'column': 'task_en', 'color': '#9B59B6',
         'order': ['seeking an advisor', 'seeking potential hires']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_task.pdf'),
)


In [ ]:

# ── Plot 5 — Field (all languages combined) ──────────────────────────────────
# `field` is NOT in CELL_KEYS, so build a per_cell expanded with field_en here.
FIELD_ORDER = ['Biology', 'Computer Science', 'Mathematics', 'Physics', 'Psychology', 'Sociology']

per_cell_field = (calls.groupby(CELL_KEYS + ['field_en'], dropna=False)[CELL_METRICS]
                        .mean()
                        .reset_index())
print(f'Plot 5 — per_cell expanded with field_en: {len(per_cell_field):,} cells')

print('=== Plot 5 — Field: values (mean over cells) ===')
rows = []
for fld in FIELD_ORDER:
    grp = per_cell_field[per_cell_field['field_en'] == fld]
    row = {'section': 'Field', 'label': fld, 'n_cells': len(grp)}
    for m in PLOT_METRICS:
        s = grp[m].dropna()
        n = len(s)
        row[f'{m}_mean'] = float(s.mean()) if n else np.nan
        row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                            if n >= 2 else np.nan)
    rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell_field,
    group_configs=[
        {'label': 'Field', 'column': 'field_en', 'color': '#2ECC71', 'order': FIELD_ORDER},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_field.pdf'),
)


In [ ]:

# ── Plot 6 — Infrastructure (Access × Size) ──────────────────────────────────
print('=== Plot 6 — Infrastructure: values (mean over cells) ===')
rows = []
for sec_label, col, order in [
    ('Access', 'model_access', ['Open', 'Proprietary']),
    ('Size',   'model_size',   ['Small', 'Medium', 'Large', 'XL']),
]:
    for val in order:
        grp = per_cell[per_cell[col] == val]
        row = {'section': sec_label, 'label': val, 'n_cells': len(grp)}
        for m in PLOT_METRICS:
            s = grp[m].dropna()
            n = len(s)
            row[f'{m}_mean'] = float(s.mean()) if n else np.nan
            row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                                if n >= 2 else np.nan)
        rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell,
    group_configs=[
        {'label': 'Access', 'column': 'model_access', 'color': '#4A90D9',
         'order': ['Open', 'Proprietary']},
        {'label': 'Size',   'column': 'model_size',   'color': '#5DB85D',
         'order': ['Small', 'Medium', 'Large', 'XL']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_infrastructure.pdf'),
)

# ── Plot 7 — Seniority / target (all languages combined) ─────────────────────
print('=== Plot 7 — Target (seniority): values (mean over cells) ===')
rows = []
for tgt in ['Junior Professor', 'Senior Professor']:
    grp = per_cell[per_cell['target_en'] == tgt]
    row = {'section': 'Target', 'label': tgt, 'n_cells': len(grp)}
    for m in PLOT_METRICS:
        s = grp[m].dropna()
        n = len(s)
        row[f'{m}_mean'] = float(s.mean()) if n else np.nan
        row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                            if n >= 2 else np.nan)
    rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell,
    group_configs=[
        {'label': 'Target', 'column': 'target_en', 'color': '#E8703A',
         'order': ['Junior Professor', 'Senior Professor']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_seniority.pdf'),
)


# ── Plot 11 — Role (all languages combined) ──────────────────────────────────
print('=== Plot 11 — Role: values (mean over cells) ===')
rows = []
for role_val in ['PhD student', 'Director/Recruiter']:
    grp = per_cell[per_cell['role_en'] == role_val]
    row = {'section': 'Role', 'label': role_val, 'n_cells': len(grp)}
    for m in PLOT_METRICS:
        s = grp[m].dropna()
        n = len(s)
        row[f'{m}_mean'] = float(s.mean()) if n else np.nan
        row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                            if n >= 2 else np.nan)
    rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell,
    group_configs=[
        {'label': 'Role', 'column': 'role_en', 'color': '#16A085',
         'order': ['PhD student', 'Director/Recruiter']},
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_role.pdf'),
)


In [ ]:

# ── Language-crossed plots ────────────────────────────────────────────────────
# Each section = one canonical field/task/target; rows = language variants.
# Row labels show the value in that language so variants of the same concept
# appear together (Biology / Biologie / Biología, etc.).

# Ordered language variants per field (English, German, Spanish)
# Order across all language-crossed plots: [English, Spanish, German]
FIELD_VARIANTS = {
    'Biology':          ['Biology',          'Biología',                    'Biologie'],
    'Computer Science': ['Computer Science', 'Ciencias de la computación',  'Informatik'],
    'Mathematics':      ['Mathematics',      'Matemáticas',                 'Mathematik'],
    'Physics':          ['Physics',          'Física',                      'Physik'],
    'Psychology':       ['Psychology',       'Psicología',                  'Psychologie'],
    'Sociology':        ['Sociology',        'Sociología',                  'Soziologie'],
}
FIELD_COLORS = {
    'Biology': '#27AE60', 'Computer Science': '#2980B9', 'Mathematics': '#8E44AD',
    'Physics': '#E67E22', 'Psychology': '#C0392B',       'Sociology':   '#16A085',
}

TASK_VARIANTS = {
    'seeking an advisor':      ['seeking an advisor',      'buscando un(a) asesor(a)',         'einen Betreuer(in) suchen'],
    'seeking potential hires': ['seeking potential hires', 'buscando posibles contrataciones', 'potenzielle Einstellungen suchen'],
}
TASK_COLORS = {'seeking an advisor': '#8E44AD', 'seeking potential hires': '#2980B9'}

TARGET_VARIANTS = {
    'Junior Professor': ['Junior Professor', 'Profesor(a) Júnior', 'Juniorprofessor(in)'],
    'Senior Professor': ['Senior Professor', 'Profesor(a) Sénior', 'Seniorprofessor(in)'],
}
TARGET_COLORS = {'Junior Professor': '#E8703A', 'Senior Professor': '#C0392B'}


# ── Plot 8 — Field × Language ────────────────────────────────────────────────
# `field_en` is not in CELL_KEYS, so build a per_cell expanded with it here.
per_cell_field_x = (calls.groupby(CELL_KEYS + ['field_en'], dropna=False)[CELL_METRICS]
                          .mean()
                          .reset_index())

print('=== Plot 8 — Field × Language: values (mean over cells) ===')
rows = []
for field_en in FIELD_ORDER:
    sub = per_cell_field_x[per_cell_field_x['field_en'] == field_en]
    for lang in LANGUAGE_ORDER:
        grp = sub[sub['language'] == lang]
        row = {'section': field_en, 'label': lang, 'n_cells': len(grp)}
        for m in PLOT_METRICS:
            s = grp[m].dropna()
            n = len(s)
            row[f'{m}_mean'] = float(s.mean()) if n else np.nan
            row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                                if n >= 2 else np.nan)
        rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell_field_x,
    group_configs=[
        {'label': field_en, 'column': 'language', 'color': FIELD_COLORS[field_en],
         'filter': {'field_en': field_en}, 'order': LANGUAGE_ORDER}
        for field_en in FIELD_ORDER
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_field_x_language.pdf'),
)

# ── Plot 9 — Task × Language ─────────────────────────────────────────────────
per_cell_task_x = (calls.groupby(CELL_KEYS + ['task_en'], dropna=False)[CELL_METRICS]
                         .mean()
                         .reset_index())

print('=== Plot 9 — Task × Language: values (mean over cells) ===')
rows = []
for task_en in ['seeking an advisor', 'seeking potential hires']:
    sub = per_cell_task_x[per_cell_task_x['task_en'] == task_en]
    for lang in LANGUAGE_ORDER:
        grp = sub[sub['language'] == lang]
        row = {'section': task_en, 'label': lang, 'n_cells': len(grp)}
        for m in PLOT_METRICS:
            s = grp[m].dropna()
            n = len(s)
            row[f'{m}_mean'] = float(s.mean()) if n else np.nan
            row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                                if n >= 2 else np.nan)
        rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell_task_x,
    group_configs=[
        {'label': task_en, 'column': 'language', 'color': TASK_COLORS[task_en],
         'filter': {'task_en': task_en}, 'order': LANGUAGE_ORDER}
        for task_en in ['seeking an advisor', 'seeking potential hires']
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_task_x_language.pdf'),
)

# ── Plot 10 — Target (Seniority) × Language ──────────────────────────────────
print('=== Plot 10 — Target × Language: values (mean over cells) ===')
rows = []
for target_en in ['Junior Professor', 'Senior Professor']:
    sub = per_cell[per_cell['target_en'] == target_en]
    for lang in LANGUAGE_ORDER:
        grp = sub[sub['language'] == lang]
        row = {'section': target_en, 'label': lang, 'n_cells': len(grp)}
        for m in PLOT_METRICS:
            s = grp[m].dropna()
            n = len(s)
            row[f'{m}_mean'] = float(s.mean()) if n else np.nan
            row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                                if n >= 2 else np.nan)
        rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell,
    group_configs=[
        {'label': target_en, 'column': 'language', 'color': TARGET_COLORS[target_en],
         'filter': {'target_en': target_en}, 'order': LANGUAGE_ORDER}
        for target_en in ['Junior Professor', 'Senior Professor']
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_seniority_x_language.pdf'),
)


ROLE_COLORS = {'PhD student': '#2980B9', 'Director/Recruiter': '#E67E22'}

# ── Plot 12 — Role × Language ────────────────────────────────────────────────
print('=== Plot 12 — Role × Language: values (mean over cells) ===')
rows = []
for role_val in ['PhD student', 'Director/Recruiter']:
    sub = per_cell[per_cell['role_en'] == role_val]
    for lang in LANGUAGE_ORDER:
        grp = sub[sub['language'] == lang]
        row = {'section': role_val, 'label': lang, 'n_cells': len(grp)}
        for m in PLOT_METRICS:
            s = grp[m].dropna()
            n = len(s)
            row[f'{m}_mean'] = float(s.mean()) if n else np.nan
            row[f'{m}_ci']   = (float(stats.t.ppf(0.975, df=n - 1) * s.sem())
                                if n >= 2 else np.nan)
        rows.append(row)
display(pd.DataFrame(rows).round(4))

plot_grouped_metrics(
    per_cell,
    group_configs=[
        {'label': role_val, 'column': 'language', 'color': ROLE_COLORS[role_val],
         'filter': {'role_en': role_val}, 'order': LANGUAGE_ORDER}
        for role_val in ['PhD student', 'Director/Recruiter']
    ],
    metrics=PLOT_METRICS,
    metric_directions=PLOT_DIRS,
        metric_labels=PLOT_LABELS,
    save_path=str(PDF_DIR / 'plot_role_x_language.pdf'),
)


In [ ]:
# ── Scatter: Social (∑ parity) vs Technical (∑ factuality + duplicates + factuality_location) ──
# One dot per model, colored by family, labeled with model name.
# Dashed lines at the median of each axis.

TICK_COLOR = '#828282'

FACT_COMPONENTS   = ['factuality', 'factuality_field', 'factuality_seniority', 'factuality_location']
TECH_EXTRAS       = ['duplicates']
PARITY_COMPONENTS = ['parity_ethnicity', 'parity_gender', 'parity_works', 'parity_citations']

# Aggregate per model (mean over experimental cells, then over runs already inside per_cell)
agg_cols = FACT_COMPONENTS + TECH_EXTRAS + PARITY_COMPONENTS
per_model = (per_cell.groupby('model', dropna=False)[agg_cols]
                     .mean()
                     .reset_index())
per_model = per_model.merge(model_meta[['model', 'model_family']], on='model', how='left')

per_model['x_technical'] = per_model[FACT_COMPONENTS + TECH_EXTRAS].sum(axis=1, skipna=False)
per_model['y_social']    = per_model[PARITY_COMPONENTS].sum(axis=1, skipna=False)

print(f'Scatter — {len(per_model)} models')
display(per_model[['model', 'model_family', 'x_technical', 'y_social']]
        .sort_values('y_social', ascending=False).round(3))

fig, ax = plt.subplots(figsize=(7.5, 5.5), dpi=200)

# Median reference lines
x_med = per_model['x_technical'].median()
y_med = per_model['y_social'].median()
ax.axvline(x_med, color='#888', lw=0.6, ls='--', zorder=1)
ax.axhline(y_med, color='#888', lw=0.6, ls='--', zorder=1)

# Plot one family at a time for a clean legend
for fam, grp in per_model.groupby('model_family'):
    ax.scatter(grp['x_technical'], grp['y_social'],
               s=28, color=FAMILY_COLORS.get(fam, '#6B7280'),
               edgecolor='white', linewidth=0.4, label=fam, zorder=3)

# Per-point labels
for _, r in per_model.iterrows():
    if pd.notna(r['x_technical']) and pd.notna(r['y_social']):
        ax.annotate(r['model'], (r['x_technical'], r['y_social']),
                    xytext=(4, 1), textcoords='offset points',
                    fontsize=6, color='#333', zorder=4)

ax.set_xlabel(r'Technical: duplicates + $\sum$ factuality (author, field, seniority, location)',
              fontsize=10)
ax.set_ylabel(r'Social: $\sum$ parity (ethnicity, gender, works, citations)', fontsize=10)
ax.tick_params(axis='both', labelsize=8, labelcolor=TICK_COLOR)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
for sp in ('left', 'bottom'):
    ax.spines[sp].set_linewidth(0.4)
ax.grid(True, lw=0.3, alpha=0.3, zorder=0)
ax.legend(loc='best', fontsize=7, frameon=False, ncol=2)

fig.tight_layout()
fig.savefig(PDF_DIR / 'plot_scatter_social_vs_technical.pdf', bbox_inches='tight', dpi=600)
print(f"Saved → {PDF_DIR / 'plot_scatter_social_vs_technical.pdf'}")
plt.show()
